<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# List Slices

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook covers:** This notebook demonstrates how to list and view your FABRIC slices using various output formats and filtering options. You will learn to display slices as tables, DataFrames, JSON, and Python lists, and how to apply filters and custom styling.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. List all your slices with `fablib.list_slices()`
2. **Select specific fields** to display a focused view
3. **Filter slices** by state or other attributes using lambda functions
4. Get output as **Pandas DataFrame**, **text**, **JSON**, or **Python list**
5. **Customize the display** with color-coded state indicators

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you should:

1. Have completed the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Ideally have one or more **existing slices** to list (create one with [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb))

**Tip:** If you have no slices, `list_slices()` will simply return an empty table. The output format examples will still work, but with no data to display.

</div>

## Background: Slice States and Naming

Each slice has a **state** that tells you where it is in its lifecycle:

| State | Meaning |
|-------|---------|
| `StableOK` | Slice is active and healthy |
| `Configuring` | Slice is being provisioned |
| `Modifying` | Slice is being modified |
| `ModifyOK` | Modification completed successfully |
| `StableError` | Slice has encountered an error |
| `Closing` | Slice is being deleted |
| `Dead` | Slice has been fully deleted |

Active slice names are **unique per user** -- you cannot have two active slices with the same name. However, you can reuse names from deleted or failed slices. If you need to reference a deleted slice, use its **slice ID** (a UUID) which is always unique.

The `list_slices()` method supports multiple **output formats**:



---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
# Import the FABlib library
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration to verify everything is set up
fablib.show_config();

---

## List All Slices

The simplest usage -- call `list_slices()` with no arguments to see all your active slices displayed as a styled table in Jupyter.

In [ ]:
# List all slices -- displays a styled table with all columns
fablib.list_slices();

---

## Select Specific Fields

Use the `fields` parameter to display only the columns you care about. This is useful when you want a concise view.

In [ ]:
# Show only the slice name and state columns
fablib.list_slices(fields=['name','state']);

---

## Filter by Values

Use the `filter_function` parameter with a **lambda function** to show only slices matching a condition. The lambda receives a dictionary of slice attributes and should return `True` for slices you want to keep.

In [ ]:
# Filter to show only slices in the 'StableOK' state
# The lambda receives each slice as a dict and returns True/False
fablib.list_slices(filter_function=lambda x: x['state'] == 'StableOK' );

---

## Output as Pandas DataFrame

Set `output='pandas'` to get a `pandas.DataFrame` object that you can manipulate programmatically -- filter rows, compute statistics, export to CSV, etc.

In [ ]:
# Get slices as a pandas DataFrame for programmatic use
output_dataframe = fablib.list_slices(output='pandas')

---

## Output as Tabular Text

Set `output='text'` to get a plain-text formatted table string. Useful for logging, printing to files, or environments without rich display.

In [ ]:
# Get slices as a plain-text table string
output_table_string = fablib.list_slices(output='text')

---

## Output as JSON

Set `output='json'` to get a JSON-formatted string. Useful for integration with other tools, APIs, or saving structured data.

In [ ]:
# Get slices as a JSON string
output_json = fablib.list_slices(output='json')

---

## Output as Python List of Dictionaries

Set `output='list'` to get a Python `List[Dict]` -- each slice is a dictionary with key-value pairs. This is the most flexible format for custom processing in Python.

In [ ]:
# Get slices as a Python list of dictionaries
output_list = fablib.list_slices(output='list');

### Use the List Programmatically

Combine `output='list'` with `quiet=True` to suppress the default display, then iterate over the results yourself.

In [ ]:
# Get the list without displaying anything (quiet=True suppresses output)
output_list = fablib.list_slices(output='list', quiet=True)

# Iterate over each slice dictionary and print selected fields
for slice in output_list:
    print(f"Slice: {slice['id']}, {slice['name']}, {slice['state']}")

---

## Add Colors to Pandas DataFrames

The `list_slices()` method can return a `pandas.DataFrame`. You can apply **conditional styling** to make slice states visually distinct -- green for healthy, yellow for in-progress, red for errors.

<div class="fab-warning">

**Note:** This example will raise an exception if you currently have no slices. Create at least one slice before running this cell.

</div>

In [ ]:
import pandas as pd
from IPython.display import clear_output


# Define a function that returns a CSS background-color based on the slice state
def state_color(val):
    if val == 'StableOK':
        color = f'{fablib.SUCCESS_LIGHT_COLOR}'          # Green for healthy
    elif val == 'Configuring' or val == 'Modifying' or val == 'ModifyOK':
        color = f'{fablib.IN_PROGRESS_LIGHT_COLOR}'      # Yellow for in-progress
    elif val == 'StableError':
        color = f'{fablib.ERROR_LIGHT_COLOR}'            # Red for errors
        
    else:
        color = ''
    return 'background-color: %s' % color


clear_output(wait=True)

# Get slices as a pandas DataFrame (quiet=True to suppress default display)
pandas_dataframe = fablib.list_slices(output='pandas', quiet=True)

# Apply the color function to the 'State' column only
pandas_dataframe = pandas_dataframe.applymap(state_color, subset=pd.IndexSlice[:, ['State']]) 
    
# Display the styled DataFrame
display(pandas_dataframe)

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Empty table returned | No active slices exist | Create a slice first using [Hello, FABRIC](../hello_fabric/hello_fabric.ipynb) |
| `filter_function` returns empty | No slices match the filter condition | Check your filter logic; try `list_slices()` without a filter first |
| Color styling raises exception | No slices exist (empty DataFrame) | Ensure you have at least one active slice |
| `applymap` deprecation warning | Newer pandas version | Replace `applymap` with `map` for pandas >= 2.1 |
| `show_config()` shows missing values | Environment not configured | Run [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) |

## FABlib API Reference

The following FABlib methods were used in this notebook:

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.show_config()` | Display current FABlib configuration | [show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config) |
| `fablib.list_slices()` | List all slices with various output options | [list_slices](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_slices) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **Showing Slice Details** | [show_slice](./show_slice.ipynb) | Inspect individual slice properties and output formats |
| **Listing Nodes & Networks** | [list_node_and_networks](./list_node_and_networks.ipynb) | View nodes, networks, interfaces, and components in a slice |
| **Creating Slices** | [create_slice](../create_slice/create_slice.ipynb) | Create slices with various submission options |
| **Deleting Slices** | [delete_slice](../delete_slice/delete_slice.ipynb) | Multiple ways to delete slices |
| **Renewing Slices** | [renew_slice](../renew_slice/renew_slice.ipynb) | Extend your slice's lease end date |